In [1]:
from dotenv import load_dotenv

In [ ]:
# Remember to do this!
load_dotenv(override=True)

In [ ]:
import os

ollama_api_key = os.getenv('OLLAMA_API_KEY')

if ollama_api_key:
    print(f"Ollama API Key exists and begins {ollama_api_key[:8]}")
else:
    print("Ollama API Key not set - please head to the troubleshooting guide in the setup folder")

In [4]:
request = "Come up with a challenging question that can be asked to LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
import ollama

response = ollama.chat(
    model="llama3.2",
    messages=messages
)

question = response.message.content

print(question)

In [8]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:
from IPython.display import Markdown, display

model_name = "llama3.2"

response = ollama.chat(model=model_name, messages=messages)
answer = response.message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "codellama"

response = ollama.chat(model=model_name, messages=messages)
answer = response.message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
print(competitors)
print(answers)

In [ ]:
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

judge_response = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": judge}])
judge_response.message.content

In [ ]:
print(judge)

In [16]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
response = ollama.chat(
    model="llama3.2",
    messages=judge_messages
)

results = response.message.content

print(results)

In [ ]:
import json

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    if result is None:
        continue
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")